# Loading and Exploring Data

This notebook will guide you through loading and exploring some of the demonstration datasets we'll be using in this course.

> 💡 At the end, there is a short exercise for you to practice loading and exploring your own dataset.



<img src="01_preprocessing.png" alt="Preprocessing diagram" style="max-width: 300px;">

These datasets include:

- The [Large Movie Review Dataset](https://ai.stanford.edu/~amaas/data/sentiment/) - a collection of 50,000 labeled IMDB reviews for binary sentiment classification.
- The [NELA-PS dataset](https://dataverse.harvard.edu/dataset.xhtml?persistentId=doi:10.7910/DVN/YHWTFC) -  a collection of 'pink slime' partisan news articles (from which we will use a small sample).
- The [WSJ-Reuters Financial News dataset](https://github.com/Finance-And-ML/News-Article-And-Full-Details-Dataset) - a collection of financial news from the Wall Street Journal and Reuters.
- The [fetch_20newsgroups dataset](https://scikit-learn.org/0.19/datasets/twenty_newsgroups.html) - a collection of approximately 20,000 newsgroup documents, organized into 20 different newsgroups.


In addition, you can load and explore your own dataset at the end of the notebook.

If you do not yet have a dataset, you might consider taking a look at [kaggle](www.kaggle.com/datasets) or [huggingface](https://huggingface.co/datasets), or the [Harvard Dataverse](https://dataverse.harvard.edu/) to find one that interests you.


## Setup

> 💡 We're going to start by loading in all the packages we'll need for this. This is best practice and will help keep our code organized.

In [ ]:
#Let's get starting importing the necessary libraries.
#The reason that we import these at the start is so that we can see if there are any issues with our environment before we start running any code.
#...it also makes it easier to keep track of what packages we are using (and avoid loading the same package multiple times).

import os #allows python to talk to the operating system (e.g., to get filepaths).
import json #we'll need this for reading in .json files (dictionaries)
import pandas as pd #pandas will allow us to work with dataframes (tabular, or .csv-like data)
from glob import glob #glob is helpful if you want to load in multiple files at once (it constructs file paths).

#If everything imported successfully, you should see no error messages! Else, you may need to install some packages (% pip install package-name).

#Let's also set a variable that points to the folder where our data is stored:
datadir = '/Users/rupertkiddle/Desktop/teach/2025/IMLfTAwP (GESIS)/3_datasets/'

## 1. Large Movie Review Dataset (IMDB)

### loading the data

In [ ]:
#first, let's define where this dataset is stored on your computer:
datadir = '/Users/rupertkiddle/Desktop/teach/2025/IMLfTAwP (GESIS)/3_datasets/aclImdb/'

> 💡 If you take a look at this dataset in your file explorer, you will notice that it is made up of a lot of .txt files that are arranged in a nested folder structure, where the folders represent the labels (positive or negative sentiment) and also the data splits (train or test). In cases like this, `glob` is very useful for loading files - so we'll use it below. 

In [ ]:
#Now, we're going to write a loop that loads in all of the txt files in the dataset. 
#Rather than point towards each individual file, we're going to use the glob package to find all of the files that match a certain pattern.

imdb_lists = [] #first we create an empty list to store our data in.

#for each file that matches the pattern (any .txt file in the pos or neg folders in train or test):
for file in glob(os.path.join(datadir, "train", "pos", "*.txt")) + \
           glob(os.path.join(datadir, "train", "neg", "*.txt")) + \
           glob(os.path.join(datadir, "test", "pos", "*.txt")) + \
           glob(os.path.join(datadir, "test", "neg", "*.txt")):
    #we're going to read the file and store its contents in a variable called 'text':
    #(notice how we are insdide the 'for' loop here, as per the indentation, meaning this will iterate over every file found by glob)
    with open(file, "r", encoding="utf-8") as f:
        text = f.read()
    #we also want to extract the batch (train/test) and label (pos/neg) from the file path:
    batch = "train" if "train" in file else "test"
    label = "pos" if "pos" in file else "neg"
    #now we append the features we just obtained, containing the batch, text, and label to our data_lists variable:
    imdb_lists.append([batch, text, label])

#The result is a list of lists, where each inner list contains the batch, text, and label for one review. 
imdb_lists[:3] #let's look at the first three entries in our list of lists.

### exploring the data (as a list)

In [ ]:
#How many samples do we have in total? (hint: what is the length?)

#Write a loop or list comprehension that counts how many samples there are of each label (pos/neg) and batch (train/test).

#Write a loop or list comprehension that returns a test_list and train_list, containing only the test and train samples respectively.

#Would you want to use a tuple instead of a list? Where? Why/why not?


In [ ]:
#A short introduction to the very useful .zip() function: 

#Sometimes, you may want to convert nested lists (like these) into separate lists: 
batches, texts, labels = zip(*imdb_lists) #the * operator tells .zip() to 'unpack' the nested lists into separate lists, which you must name.

#Now you have three lists, so that you can access the batches, texts, and labels separately:
print(batches[:3]) #first three batches
print(texts[:3]) #first three texts
print(labels[:3]) #first three labels

#Putting them back together again is just as easy:
imdb_lists = list(zip(batches, texts, labels)) 

> 💡 With a dataset as simple as this, you would often not bother to cast it into any more complex data structures like a dictionary or a dataframe. 

> However, let's at least explore how a dataframe can be useful for inspecting your data:

### exploring the data (as a dataframe)

In [ ]:
#It an be useful to get a 'birds eye view' of the data by converting it to a dataframe:
imdb_df = pd.DataFrame(imdb_lists, columns=["batch", "text", "label"]) #we pass in the data (imdb_lists) and also the column names (batch, text, label)

#Now let's look at the first few rows of the dataframe:
imdb_df.head() #.head() shows the first five rows of the dataframe.

#NOTE: under the hood, our lists have now been converted into pd.Series objects, which are like lists but with a lot of extra functionality (all tied to Pandas).

In [ ]:
#Pandas can give us some quick summary statistics about the dataframe:
imdb_df.describe() #.describe() gives us some summary statistics about the dataframe.

In [ ]:
#Pandas can also tell us if there are any missing values:
imdb_df.isnull().sum() #.isnull().sum() tells us how many missing values there are in each column.
#BONUS: what happens if we just run the method .isnull() without the .sum()? Why?

In [ ]:
#Pandas can even tell us the distribution of values in a column:
imdb_df['label'].value_counts() 
#NOTE: notice that here we are calling .value_counts() on a specific column of the dataframe (the 'label' column).

In [ ]:
#or the distribution of the length of the reviews:
imdb_df['text'].str.len().describe() 
#NOTE notice that we are chain-calling the len() and describe() methods here, using the .str accessor to tell pandas that we want to treat the 'text' column as strings.

### summing up what we see

**IMDB Large Movie Review Dataset Overview:**

- **Total samples**: 50,000 movie reviews from IMDB
- **Labels**: Binary classification (positive/negative sentiment)
- **Data split**: 
  - 🔸 Training set: 25,000 reviews
  - 🔸 Test set: 25,000 reviews
- **Class balance (i.e., distribution of labels)**: 
  - ✅ Training: 12,500 positive + 12,500 negative
  - ✅ Test: 12,500 positive + 12,500 negative
- **Data quality**:
  - ✅ No missing values (confirmed via `.isnull()`)
  - ✅ Minimal duplicates (confirmed via `.describe()`)

> 💡 Of course, we do not yet know how clean the text fields of the reviews are. 

## 2. NELA-PS Dataset

### loading the data

In [ ]:
#first, let's define where this dataset is stored on your computer:
datadir = '/Users/rupertkiddle/Desktop/teach/2025/IMLfTAwP (GESIS)/3_datasets/NELA-PS/'

> 💡 If you take a look at this dataset in your file explorer, you will see we have another `glob' situation. In this case we have articles nested within outlet-labeled folders nested within date-labeled folders. So, we can use a similar logic as we did for the IMDB dataset, to preserve these 'labels' when loading the articles. 

> ⚠️ If you look a little closer, you will notice that the articles do not have file extensions. While it is convention to give files extensions (txt, csv, etc), these are not actually necessary for the computer to read them. We can just use a wildcard (*) in our glob pattern to match any file.

In [ ]:
#Now, we're going to write a loop that loads in all of the txt files in the dataset. 
#Rather than point towards each individual file, we're going to use the glob package to find all of the files that match a certain pattern.

nela_lists = [] #first we create an empty list to store our data in.

#for each file that matches the pattern (any file within outlet folders within date folders):
for file in glob(os.path.join(datadir, "*", "*", "*")):
    #we're going to read the file and store its contents in a variable called 'text':
    #(notice how we are inside the 'for' loop here, as per the indentation, meaning this will iterate over every file found by glob)
    with open(file, "r", encoding="utf-8") as f:
        text = f.read()
    
    #we want to extract the date and outlet from the file path:
    path_parts = file.split(os.sep)  # Split the path by the OS separator
    date = path_parts[-3]  # Date folder (YYYY-MM-DD format), which is the 'third last' part of the path
    outlet = path_parts[-2]  # Outlet folder, which is the 'second last' part of the path
    
    #now we append the features we just obtained, containing the date, outlet, and text to our data_lists variable:
    nela_lists.append([date, outlet, text])

#The result is a list of lists, where each inner list contains the date, outlet, and text for one article. 
nela_lists[:3] #let's look at the first three entries in our list of lists.

> 💡 Great, so now we have the same data structure as we did before (a list of lists). Let's put this in a dataframe to summarize the data: 

### exploring the data

In [ ]:
#pass the list of lists to a pd.DataFrame():
nela_df = pd.DataFrame(nela_lists, columns=["date", "outlet", "text"]) #we pass in the data (nela_lists) and also the column names (date, outlet, text)

#Inspect the first few rows of the dataframe:
nela_df.head()

In [ ]:
#how many samples do we have in total? (hint: a dataframe has .shape attribute that you can call)

In [ ]:
#do any columns have missing data?

In [ ]:
#let's check the date range: 
nela_df['date'] = pd.to_datetime(nela_df['date']) #first we convert the 'date' column to datetime format (it is currently a string)
min_date = nela_df['date'].min() #get the minimum value of the 'date' column
max_date = nela_df['date'].max() #get the maximum value of the 'date' column
print(max_date.date()), print(min_date.date())

#BONUS: can you calculate the total date range in days?

In [ ]:
#A quick introduction to 'F-strings' (formatted string literals) - 

#Sometimes, you want to include variables inside strings, for example:
print(f"The dataset contains articles from {min_date.date()} to {max_date.date()}.")
#...the f before the string tells Python to interpret any variables inside curly braces {}.
#These are really useful for printing out informative messages that include the values of variables. 

In [ ]:
#what are the distributions of articles per outlet? 

In [ ]:
#what are the distribution of articles by date?

### summing up what you see

**NELA-PS Dataset Overview:**

*[Enter your observations here]*


## 3. WSJ_Reuters Financial News Dataset

> 💡 Now things get a little more interesting. If you take a look at the dataset, you will find a single file in .JSON format. These can store all kinds of data types, but often you will find they contain dictionaries (key-value pairs). Still, the only way to know, is to take a look.

### loading the data

In [ ]:
#first, let's define where this dataset is stored on your computer:
datadir = '/Users/rupertkiddle/Desktop/teach/2025/IMLfTAwP (GESIS)/3_datasets/WSJ-Reuters/'

#let's use glob to get the filepath of the .json file in the directory:
file_path = glob(os.path.join(datadir, "*.json")) 
file_path #this should return a list with a single filepath in it.


In [ ]:
#now, let's take a peek at the first two lines of the file: 
with open(file_path[0], "r", encoding="utf-8") as f:
    first_line = f.readline() #read the first line of the file
    second_line = f.readline() #read the second line of the file
print(first_line) #print the first line
print(second_line) #print the second line

#See the {key: value} pairs? This indicates that the file indeed contains data in dictionary format.

In [ ]:
#Great, so now we know the structure of the file: it contains one dictionary per line. Let's load it in to a list of dictionaries:
data_dicts = [] #create an empty list to store the dictionaries in
with open(file_path[0], "r", encoding="utf-8") as f:
    for line in f: #for each line in the file
        data_dicts.append(json.loads(line)) #load the line as a dictionary and append it to the list

#...and let's look at the first two dictionaries:
data_dicts[:2]

### exploring the data

In [ ]:
#Let's take a look at the keys in the first dictionary:
data_dicts[0].keys() #notice we are calling .keys() on the first element [0] of the list of dictionaries.

> 💡 From here, feel free to explore the data as you like. Since it appears to be 'flat' (i.e., does not have nested collections), you could convert it to a dataframe. Alternatively, you could perform analysis directly on the lists of dictionaries, or pull the values out into separate lists using a loop or list comprehension. 

In [ ]:
#your code...

In [ ]:
#CHALLENGE: Write a loop that creates a dict where the keys are the URLs and the values are dicts containing the other key-value pairs for that article.
#Your final dict should look like:
#{"url1": {'news_title': '...', 'news_time': '...', 'content': '...', 'keywords': '...'},
# "url2": {'news_title': '...', 'news_time': '...', 'content': '...', 'keywords': '...'}, ...}
#NOTE: This is a common structure to have if you scrape data from the web yourself - some ID as key (e.g., URL), and the other contained as k:v pairs in a nested dict.

> 💡 The true worth of the dict structure becomes apparent when you have deeply nested datasets. This is not something that we will encouter with the example datasets in this course, but if you do have such data yourself, and need some assistance wrangling it, feel free to ask!

### summing up what you see - 

**WSJ-Reuters Dataset Overview:**

*[Enter your observations here]*


## 4. Fetch_20newsgroups Dataset

> 💡 This dataset we will not download in the usual manner. It is a dataset that is served by the sklearn library. Nowadays, several machine learning libraries include built-in datasets for convenience. For example, take a look at [sklearn datasets](https://scikit-learn.org/stable/api/sklearn.datasets.html) and [HuggingFace datasets](https://huggingface.co/docs/datasets/en/index). For such datasets, we can retrieve them within our code. 

In [ ]:
#first we need to import the function to fetch the dataset:
from sklearn.datasets import fetch_20newsgroups

#now we can fetch the dataset (hover over the function name to see the docstring)
docs = fetch_20newsgroups()

> 💡 Take a look at sklearn's guide to the [20 newsgroups dataset](https://scikit-learn.org/stable/datasets/real_world.html#newsgroups-dataset) to get an idea of what sort of data is there and how you can pass arguments to the function above to get different parts of the dataset. Additionally, feel free to explore their [other datasets](https://scikit-learn.org/stable/api/sklearn.datasets.html) as you like.


### summing up what you see - 

**Fetch_20newsgroups Dataset:**

*[Enter your observations here]*


## 5. Your own dataset!
> 💡 Feel free to load and play around with your own dataset. If you do not yet have one, see the notes at the top of this workbook. 

In [1]:
#your code...